# Phase 2 — Feature Engineering

**Goals:**
- Apply the full preprocessing pipeline to all 4 CMAPSS variants
- Verify per-condition normalization (FD002/FD004)
- Visualize rolling feature effects
- Save processed datasets to `data/processed/`

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from preprocessing import (
    run_preprocessing_pipeline, load_raw, compute_train_rul,
    compute_rolling_features, INFORMATIVE_SENSORS, WINDOW_SIZES,
    N_CONDITIONS, RUL_CAP
)

DATA_RAW     = Path('../data/raw')
DATA_PROC    = Path('../data/processed')
MODELS_DIR   = Path('../models')
FD_IDS       = ['FD001', 'FD002', 'FD003', 'FD004']

plt.rcParams['figure.dpi'] = 110
print('Setup complete.')

## 1. Run Preprocessing Pipeline for All 4 Datasets

This will:
1. Load raw data
2. Drop constant sensors
3. Compute capped RUL (training only)
4. Fit KMeans clusters (op1/op2) per dataset
5. Fit per-condition MinMaxScalers
6. Apply scalers
7. Compute rolling features (windows: 5, 10, 30)
8. Split by engine (80/20)
9. Save parquet files + joblib artifacts

In [ ]:
artifacts = {}

for fd_id in FD_IDS:
    art = run_preprocessing_pipeline(
        raw_dir=DATA_RAW,
        output_dir=DATA_PROC,
        fd_id=fd_id,
        n_conditions=N_CONDITIONS[fd_id],
        rul_cap=RUL_CAP,
        val_fraction=0.2,
    )
    artifacts[fd_id] = art

print('\nAll datasets processed successfully!')

## 2. Verify Processed Data Quality

In [ ]:
print('Checking for NaN values and feature ranges...\n')

for fd_id in FD_IDS:
    X_train = pd.read_parquet(DATA_PROC / fd_id / 'train_features.parquet')
    y_train = pd.read_parquet(DATA_PROC / fd_id / 'train_labels.parquet')
    
    nan_count = X_train.isna().sum().sum()
    feat_min  = X_train.min().min()
    feat_max  = X_train.max().max()
    
    print(f'{fd_id}: rows={len(X_train):>6}  features={X_train.shape[1]}  '
          f'NaN={nan_count}  '
          f'feat_range=[{feat_min:.3f}, {feat_max:.3f}]  '
          f'RUL_range=[{y_train.rul.min():.0f}, {y_train.rul.max():.0f}]')

## 3. Feature Count Summary

In [ ]:
feature_cols = json.loads((DATA_PROC / 'FD001' / 'feature_cols.json').read_text())

raw_sensors   = [c for c in feature_cols if not 'roll' in c and c not in ('op1','op2')]
rolling_feats = [c for c in feature_cols if 'roll' in c]
op_feats      = [c for c in feature_cols if c in ('op1','op2')]

print(f'Raw sensors:      {len(raw_sensors)}')
print(f'Rolling features: {len(rolling_feats)}  ({len(INFORMATIVE_SENSORS)} sensors × 2 stats × {len(WINDOW_SIZES)} windows)')
print(f'Op settings:      {len(op_feats)}')
print(f'Total features:   {len(feature_cols)}')

## 4. Visualize Rolling Feature Effect

In [ ]:
# Compare raw s11 vs rolling mean at window=5, 10, 30 for one engine
from preprocessing import load_raw, drop_low_variance_sensors, compute_rolling_features, \
    fit_condition_clusters, assign_condition_labels, fit_scalers_per_condition, apply_scalers

raw = load_raw(DATA_RAW, 'FD001', 'train')
raw = drop_low_variance_sensors(raw)
raw = compute_train_rul(raw)

# Normalize first
from joblib import load as jload
km      = jload(Path('../models/FD001/kmeans.joblib'))
scalers = jload(Path('../models/FD001/scalers.joblib'))
raw = assign_condition_labels(raw, km)
raw = apply_scalers(raw, scalers, INFORMATIVE_SENSORS + ['op1','op2'])
raw = compute_rolling_features(raw)

uid   = 1
engine = raw[raw.unit_id == uid].sort_values('cycle')
sensor = 's11'

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(engine.cycle, engine[sensor], alpha=0.4, lw=1, color='gray', label='Raw s11')
for w in WINDOW_SIZES:
    col = f'{sensor}_roll{w}_mean'
    ax.plot(engine.cycle, engine[col], lw=2, label=f'Roll mean w={w}')

ax.set_title(f'Engine {uid} — s11 Raw vs Rolling Mean (normalized)')
ax.set_xlabel('Cycle')
ax.set_ylabel('Normalized value')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Per-Condition Normalization — FD002 Sanity Check

In [ ]:
X_fd002 = pd.read_parquet(DATA_PROC / 'FD002' / 'train_features.parquet')
meta_fd002 = pd.read_parquet(DATA_PROC / 'FD002' / 'train_meta.parquet')

# Sensor value distribution should be [0,1] per condition
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

raw_fd002 = load_raw(DATA_RAW, 'FD002', 'train')
axes[0].hist(raw_fd002['s7'].values, bins=50, color='steelblue', edgecolor='k', alpha=0.7)
axes[0].set_title('FD002 s7 RAW (bimodal from 6 conditions)')
axes[0].set_xlabel('Raw sensor value')

axes[1].hist(X_fd002['s7'].values, bins=50, color='green', edgecolor='k', alpha=0.7)
axes[1].set_title('FD002 s7 NORMALIZED (per-condition)')
axes[1].set_xlabel('Normalized value')

plt.tight_layout()
plt.show()
print('After per-condition normalization, sensor values should be uniformly distributed in [0, 1].')

## 6. Saved Artifacts Summary

In [ ]:
import os

print('Processed data files:')
for fd_id in FD_IDS:
    print(f'\n  {fd_id}/')
    for f in sorted((DATA_PROC / fd_id).glob('*')):
        size_kb = f.stat().st_size / 1024
        print(f'    {f.name:<35} {size_kb:>8.1f} KB')

print('\nModel artifacts:')
for fd_id in FD_IDS:
    print(f'\n  {fd_id}/')
    for f in sorted((Path('../models') / fd_id).glob('*')):
        size_kb = f.stat().st_size / 1024
        print(f'    {f.name:<35} {size_kb:>8.1f} KB')

## Summary

Feature engineering complete. For each of the 4 datasets:
- **100 features** per row (14 raw + 84 rolling + 2 op settings)
- **No NaN values** (rolling NaN resolved via ffill/bfill per engine)
- **All sensor values in [0, 1]** after per-condition normalization
- **Engine-level train/val split** (no temporal leakage)
- **Artifacts saved**: kmeans.joblib, scalers.joblib, feature_cols.json

→ Proceed to `03_modeling.ipynb`